# 02 · Reference fingerprint
Computes Tiny Aya Global's refusal-vector fingerprint on split A, then on split B, and checks they agree (cosine close to 1). Logic in `src/forged/fingerprint.py`.

Needs the contrast sets from `01` in `data/contrast_sets/public` (rerun 01, or download them from your HF repo).

In [ ]:
# Clone the private repo and install the package (+ Lexsi libraries).
# Needs a GitHub token with read access stored as a Colab secret named GH_TOKEN.
import os, subprocess
from google.colab import userdata
os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
REPO = 'mathcrypto/forged-fingerprints'
if not os.path.exists('/content/forged-fingerprints'):
    subprocess.run(['git', 'clone', f'https://{os.environ["GH_TOKEN"]}@github.com/{REPO}.git', '/content/forged-fingerprints'], check=True)
else:
    subprocess.run(['git', '-C', '/content/forged-fingerprints', 'pull'], check=True)
%cd /content/forged-fingerprints

In [ ]:
from huggingface_hub import snapshot_download
HF_ORG = 'YOUR-HF-ORG'   # <- change me
snapshot_download(f'{HF_ORG}/forged-fingerprints-contrast', repo_type='dataset', local_dir='data/contrast_sets/public')

In [ ]:
!python scripts/compute_fingerprint.py --config configs/fingerprint.yaml --split A
!python scripts/compute_fingerprint.py --config configs/fingerprint.yaml --split B --compare outputs/fingerprints/tiny-aya-global_A.pt

## Per-layer view
Which layers carry the refusal direction most consistently between A and B.

In [ ]:
from forged.fingerprint import Fingerprint, similarity
a = Fingerprint.load('outputs/fingerprints/tiny-aya-global_A.pt')
b = Fingerprint.load('outputs/fingerprints/tiny-aya-global_B.pt')
sim = similarity(a, b)
print('aggregate cosine:', round(sim['cosine'], 4))
for l, c in sim['per_layer'].items(): print(f'layer {l:2d}: {c:.4f}')